#### Importing Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 6)

#### Loading the Dataset

In [3]:
data_path = Path("../data/raw/Online Retail.xlsx")

df = pd.read_excel(data_path, engine="openpyxl")

print(f"Dataset Shape: {df.shape}")
print(f"Date Range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")

display(df.head())

Dataset Shape: (541909, 8)
Date Range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,"17,850.00",United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,"17,850.00",United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom


#### Saving the Raw Dataset in Parquet Format

In [6]:
parquet_path = Path("../data/processed/online_retail_raw.parquet")
parquet_path.parent.mkdir(parents=True, exist_ok=True)

text_columns = ["InvoiceNo", "StockCode", "Description", "Country"]

df[text_columns] = df[text_columns].astype("string")

df.to_parquet(parquet_path, engine="pyarrow", index=False)

print(f"Parquet file saved to: {parquet_path}")
print(f"File size: {parquet_path.stat().st_size / (1024 ** 2):.2f} MB")

print("\nDataset Shape:")
print(df.shape)

print("\nColumn Data Types:")
display(df.dtypes.to_frame("Data Type"))

print("\nMissing Values:")
display(df.isna().sum().to_frame("Missing Values"))

Parquet file saved to: ..\data\processed\online_retail_raw.parquet
File size: 3.42 MB

Dataset Shape:
(541909, 8)

Column Data Types:


,Data Type
InvoiceNo,string
StockCode,string
Description,string
Quantity,int64
InvoiceDate,datetime64[us]
UnitPrice,float64
CustomerID,float64
Country,string



Missing Values:


,Missing Values
InvoiceNo,0
StockCode,0
Description,1454
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,135080
Country,0


#### Creating Transaction-Level Revenue Variables

In [7]:
eda_df = df.copy()

eda_df["Transaction_Date"] = eda_df["InvoiceDate"].dt.normalize()
eda_df["Line_Amount"] = eda_df["Quantity"] * eda_df["UnitPrice"]
eda_df["Is_Cancellation"] = eda_df["InvoiceNo"].str.startswith("C")

eda_df["Transaction_Type"] = np.select(
    [
        eda_df["Is_Cancellation"],
        eda_df["Quantity"] < 0,
        eda_df["UnitPrice"] < 0,
        eda_df["UnitPrice"] == 0
    ],
    [
        "Cancellation",
        "Other Negative Quantity",
        "Negative Unit Price",
        "Zero Unit Price"
    ],
    default="Positive Quantity and Price"
)

display(
    eda_df[
        [
            "InvoiceNo",
            "StockCode",
            "Transaction_Date",
            "Quantity",
            "UnitPrice",
            "Line_Amount",
            "Is_Cancellation",
            "Transaction_Type"
        ]
    ].head(10)
)

print("\nTransaction Type Distribution:")
display(
    eda_df["Transaction_Type"]
    .value_counts()
    .rename_axis("Transaction_Type")
    .reset_index(name="Records")
)

,InvoiceNo,StockCode,Transaction_Date,Quantity,UnitPrice,Line_Amount,Is_Cancellation,Transaction_Type
0,536365,85123A,2010-12-01,6,2.55,15.30,False,Positive Quantity and Price
1,536365,71053,2010-12-01,6,3.39,20.34,False,Positive Quantity and Price
2,536365,84406B,2010-12-01,8,2.75,22.00,False,Positive Quantity and Price
3,536365,84029G,2010-12-01,6,3.39,20.34,False,Positive Quantity and Price
4,536365,84029E,2010-12-01,6,3.39,20.34,False,Positive Quantity and Price
5,536365,22752,2010-12-01,2,7.65,15.30,False,Positive Quantity and Price
6,536365,21730,2010-12-01,6,4.25,25.50,False,Positive Quantity and Price
7,536366,22633,2010-12-01,6,1.85,11.10,False,Positive Quantity and Price
8,536366,22632,2010-12-01,6,1.85,11.10,False,Positive Quantity and Price
9,536367,84879,2010-12-01,32,1.69,54.08,False,Positive Quantity and Price



Transaction Type Distribution:


,Transaction_Type,Records
0,Positive Quantity and Price,530104
1,Cancellation,9288
2,Other Negative Quantity,1336
3,Zero Unit Price,1179
4,Negative Unit Price,2


**Key Observations**
- **Positive-quantity, positive-price transactions account for 530,104 records (97.82%)**, confirming that conventional sales transactions dominate the dataset.
- **Cancellation invoices account for 9,288 records (1.71%)**, highlighting the importance of distinguishing returns and cancellations from completed sales when calculating revenue.
- **An additional 1,336 records contain negative quantities outside cancellation invoices**, while 1,179 records have zero unit prices and only 2 contain negative unit prices. These entries require separate treatment during revenue construction.
- **Transaction classification provides a foundation for distinguishing merchandise sales from financial adjustments**, but transaction type alone cannot identify all non-merchandise entries, such as postage, fees, and discounts.

**EDA Decision**

Retain all transaction categories during exploratory analysis and compare their financial contributions before establishing the final daily merchandise revenue definition.

#### Analyzing Financial Contributions by Transaction Type

In [8]:
transaction_financial_summary = (
    eda_df.groupby("Transaction_Type")
    .agg(
        Transaction_Records=("Line_Amount", "size"),
        Total_Amount=("Line_Amount", "sum"),
        Average_Line_Amount=("Line_Amount", "mean"),
        Median_Line_Amount=("Line_Amount", "median")
    )
    .reset_index()
)

transaction_financial_summary["Record_Percentage"] = (
    transaction_financial_summary["Transaction_Records"]
    / len(eda_df) * 100
)

transaction_financial_summary["Contribution_to_Net_Percentage"] = (
    transaction_financial_summary["Total_Amount"]
    / eda_df["Line_Amount"].sum() * 100
)

transaction_financial_summary = transaction_financial_summary[
    [
        "Transaction_Type",
        "Transaction_Records",
        "Record_Percentage",
        "Total_Amount",
        "Contribution_to_Net_Percentage",
        "Average_Line_Amount",
        "Median_Line_Amount"
    ]
].sort_values("Total_Amount", ascending=False)

display(transaction_financial_summary)

print(
    f"\nPreliminary Net Transaction Amount: "
    f"{eda_df['Line_Amount'].sum():,.2f}"
)

,Transaction_Type,Transaction_Records,Record_Percentage,Total_Amount,Contribution_to_Net_Percentage,Average_Line_Amount,Median_Line_Amount
3,Positive Quantity and Price,530104,97.82,"10,666,684.54",109.43,20.12,9.90
4,Zero Unit Price,1179,0.22,0.00,0.00,0.00,0.00
2,Other Negative Quantity,1336,0.25,0.00,0.00,0.00,-0.00
1,Negative Unit Price,2,0.00,"-22,124.12",-0.23,"-11,062.06","-11,062.06"
0,Cancellation,9288,1.71,"-896,812.49",-9.20,-96.56,-8.50



Preliminary Net Transaction Amount: 9,747,747.93


**Key Observations**
- **Positive-quantity, positive-price transactions generate 10,666,684.54, representing 109.43% of preliminary net transaction value**, demonstrating that positive financial activity exceeds the final net amount before deductions.
- **Cancellation invoices reduce the preliminary net amount by 896,812.49 (9.20%)**, making them the largest negative financial component and an important consideration when defining gross versus net merchandise revenue.
- **Two negative-price adjustments contribute an additional reduction of 22,124.12 (0.23%)**, while zero-price transactions and other negative-quantity records have no direct monetary contribution.
- **The preliminary net transaction amount totals 9,747,747.93**, reflecting positive transactions after cancellations and negative-price adjustments. This figure is not yet a validated merchandise revenue total because it includes non-merchandise financial entries.

**EDA Decision**

Examine non-merchandise stock codes and their financial contributions before selecting the daily revenue measure. Maintain a clear distinction between gross merchandise sales, merchandise returns, and accounting adjustments to support consistent forecasting.

#### Analyzing Non-Merchandise Transaction Contributions

In [9]:
non_merchandise_codes = [
    "POST", "DOT", "C2", "M", "D", "S",
    "BANK CHARGES", "AMAZONFEE", "CRUK", "B"
]

non_merchandise_df = eda_df[
    eda_df["StockCode"].isin(non_merchandise_codes)
].copy()

non_merchandise_summary = (
    non_merchandise_df.groupby("StockCode")
    .agg(
        Transaction_Records=("InvoiceNo", "size"),
        Unique_Invoices=("InvoiceNo", "nunique"),
        Positive_Amount=("Line_Amount", lambda x: x[x > 0].sum()),
        Negative_Amount=("Line_Amount", lambda x: x[x < 0].sum()),
        Net_Amount=("Line_Amount", "sum")
    )
    .sort_values("Net_Amount", ascending=False)
)

display(non_merchandise_summary)

print("\nOverall Non-Merchandise Summary:")
print(f"Transaction Records: {len(non_merchandise_df):,}")
print(f"Positive Amount: {non_merchandise_df.loc[non_merchandise_df['Line_Amount'] > 0, 'Line_Amount'].sum():,.2f}")
print(f"Negative Amount: {non_merchandise_df.loc[non_merchandise_df['Line_Amount'] < 0, 'Line_Amount'].sum():,.2f}")
print(f"Net Amount: {non_merchandise_df['Line_Amount'].sum():,.2f}")

print("\nShare of Preliminary Net Transaction Amount:")
print(
    f"{non_merchandise_df['Line_Amount'].sum() / eda_df['Line_Amount'].sum() * 100:.2f}%"
)

,Transaction_Records,Unique_Invoices,Positive_Amount,Negative_Amount,Net_Amount
StockCode,,,,,
DOT,710,710,"206,248.77",-3.29,"206,245.48"
POST,1256,1254,"78,101.88","-11,871.24","66,230.64"
C2,144,144,"7,051.00",-65.00,"6,986.00"
S,63,19,63.05,"-3,112.44","-3,049.39"
D,77,65,0.00,"-5,696.22","-5,696.22"
BANK CHARGES,37,36,165.00,"-7,340.64","-7,175.64"
CRUK,16,16,0.00,"-7,933.43","-7,933.43"
B,3,3,"11,062.06","-22,124.12","-11,062.06"
M,571,517,"78,110.27","-146,784.46","-68,674.19"



Overall Non-Merchandise Summary:
Transaction Records: 2,911
Positive Amount: 394,563.12
Negative Amount: -440,212.43
Net Amount: -45,649.31

Share of Preliminary Net Transaction Amount:
-0.47%


In [10]:
display(non_merchandise_summary)

,Transaction_Records,Unique_Invoices,Positive_Amount,Negative_Amount,Net_Amount
StockCode,,,,,
DOT,710,710,"206,248.77",-3.29,"206,245.48"
POST,1256,1254,"78,101.88","-11,871.24","66,230.64"
C2,144,144,"7,051.00",-65.00,"6,986.00"
S,63,19,63.05,"-3,112.44","-3,049.39"
D,77,65,0.00,"-5,696.22","-5,696.22"
BANK CHARGES,37,36,165.00,"-7,340.64","-7,175.64"
CRUK,16,16,0.00,"-7,933.43","-7,933.43"
B,3,3,"11,062.06","-22,124.12","-11,062.06"
M,571,517,"78,110.27","-146,784.46","-68,674.19"


**Key Observations**
- **The ten identified non-merchandise stock codes account for 2,911 transaction records**, generating 394,563.12 in positive amounts and -440,212.43 in negative amounts, resulting in a net contribution of -45,649.31 (-0.47% of preliminary net transaction value).
- **DOT and POST contribute the largest positive net amounts, at 206,245.48 and 66,230.64, respectively**, demonstrating that postage-related charges can inflate reported merchandise revenue when included in transaction totals.
- **AMAZONFEE generates the largest negative net contribution (-221,520.50)**, followed by manual entries under stock code M (-68,674.19), highlighting the financial significance of administrative and accounting adjustments.
- **Other non-merchandise codes represent discounts, bank charges, commissions, carriage, samples, and bad-debt adjustments**, reinforcing the need to distinguish operational charges from product sales.
- **The relatively small overall net contribution conceals substantial offsetting positive and negative amounts**, meaning that evaluating non-merchandise transactions solely by their net percentage would understate their influence on revenue calculations.

**EDA Decision**

Exclude clearly identifiable non-merchandise charges and accounting adjustments from the eventual merchandise revenue target, subject to validating the transaction classifications. Investigate ambiguous manual entries separately before establishing final filtering rules.

### Univariate Analysis
#### Analyzing the Distribution of Daily Sales Revenue

In [13]:
import plotly.graph_objects as go

non_merchandise_codes = [
    "POST", "DOT", "C2", "M", "D",
    "S", "BANK CHARGES", "AMAZONFEE", "CRUK", "B"
]

merchandise_df = eda_df.loc[
    ~eda_df["StockCode"].isin(non_merchandise_codes)
].copy()

daily_revenue = (
    merchandise_df.groupby("Transaction_Date")["Line_Amount"]
    .sum()
    .sort_index()
)

mean_revenue = daily_revenue.mean()
median_revenue = daily_revenue.median()

fig = go.Figure()

fig.add_trace(
    go.Histogram(
        x=daily_revenue,
        nbinsx=30,
        name="Daily Revenue",
        marker=dict(
            color="#4C78A8",
            line=dict(color="white", width=1)
        ),
        opacity=0.85,
        hovertemplate="Daily Revenue: %{x:,.2f}<br>Number of Days: %{y}<extra></extra>"
    )
)

fig.add_vline(
    x=mean_revenue,
    line=dict(color="#E45756", width=2, dash="dash")
)

fig.add_vline(
    x=median_revenue,
    line=dict(color="#2A9D8F", width=2, dash="dash")
)

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(color="#E45756", width=2, dash="dash"),
        name=f"Mean: {mean_revenue:,.2f}"
    )
)

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(color="#2A9D8F", width=2, dash="dash"),
        name=f"Median: {median_revenue:,.2f}"
    )
)

fig.update_layout(
    title=dict(
        text="Distribution of Daily Retail Sales Revenue",
        x=0.5,
        xanchor="center"
    ),
    xaxis_title="Daily Sales Revenue",
    yaxis_title="Number of Days",
    template="plotly_white",
    width=1100,
    height=550,
    bargap=0.05,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    ),
    hovermode="closest"
)

fig.show()

display(
    daily_revenue.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    ).to_frame("Daily_Revenue")
)

,Daily_Revenue
count,305.00
mean,"32,109.50"
std,"17,382.67"
min,"3,457.11"
25%,"20,861.90"
50%,"27,871.61"
75%,"42,007.09"
90%,"55,465.13"
95%,"62,649.26"
99%,"81,072.38"


**Key Observations**
- **Daily sales revenue exhibits a positively skewed distribution**, with a mean of 32,109.50 exceeding the median of 27,871.61, indicating that higher-revenue days pull the average upward.
- **The middle 50% of trading days generate between 20,861.90 and 42,007.09 in revenue**, while the standard deviation of 17,382.67 highlights substantial variation in daily sales performance.
- **Daily revenue ranges from 3,457.11 to 110,806.17**, with the 95th and 99th percentiles reaching 62,649.26 and 81,072.38, respectively, revealing a small number of exceptionally high-revenue days.

**Business Insights**
- **Considerable variation in daily revenue highlights the importance of reliable short-term forecasting for inventory planning, staffing, and operational resource allocation.**
- **The right-skewed distribution suggests that forecasting models should be evaluated across both typical and high-revenue periods**, ensuring that strong performance on ordinary trading days does not conceal larger errors during sales peaks.

#### Analyzing Daily Sales Revenue Outliers

In [14]:
import plotly.graph_objects as go

q1 = daily_revenue.quantile(0.25)
q3 = daily_revenue.quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

revenue_outliers = daily_revenue[
    (daily_revenue < lower_bound) |
    (daily_revenue > upper_bound)
]

fig = go.Figure()

fig.add_trace(
    go.Box(
        x=daily_revenue.values,
        y=["Daily Revenue"] * len(daily_revenue),
        orientation="h",
        name="Daily Revenue",
        boxpoints="outliers",
        quartilemethod="linear",
        marker=dict(
            color="#D9534F",
            size=7,
            opacity=0.7
        ),
        line=dict(color="#377EB8", width=2),
        fillcolor="rgba(55, 126, 184, 0.35)",
        customdata=daily_revenue.index.strftime("%Y-%m-%d"),
        hovertemplate=(
            "Daily Revenue: %{x:,.2f}"
            "<extra></extra>"
        )
    )
)

fig.update_layout(
    title=dict(
        text="Distribution and Outliers in Daily Retail Sales Revenue",
        x=0.5,
        xanchor="center"
    ),
    xaxis_title="Daily Sales Revenue",
    yaxis_title="",
    template="plotly_white",
    width=1100,
    height=450,
    showlegend=False,
    font=dict(size=13)
)

fig.show()

outlier_summary = pd.DataFrame({
    "Metric": [
        "Total Trading Days",
        "Lower Outlier Bound",
        "Upper Outlier Bound",
        "Lower Outlier Days",
        "Upper Outlier Days",
        "Total Outlier Days",
        "Outlier Percentage",
        "Minimum Daily Revenue",
        "Maximum Daily Revenue",
        "Average Outlier Revenue"
    ],
    "Value": [
        len(daily_revenue),
        lower_bound,
        upper_bound,
        (daily_revenue < lower_bound).sum(),
        (daily_revenue > upper_bound).sum(),
        len(revenue_outliers),
        len(revenue_outliers) / len(daily_revenue) * 100,
        daily_revenue.min(),
        daily_revenue.max(),
        revenue_outliers.mean() if len(revenue_outliers) else np.nan
    ]
})

display(outlier_summary)

print("\nOutlier Dates and Revenue:")
display(
    revenue_outliers.sort_values(ascending=False)
    .rename("Daily_Revenue")
    .to_frame()
)

,Metric,Value
0,Total Trading Days,305.00
1,Lower Outlier Bound,"-10,855.88"
2,Upper Outlier Bound,"73,724.87"
3,Lower Outlier Days,0.00
4,Upper Outlier Days,9.00
5,Total Outlier Days,9.00
6,Outlier Percentage,2.95
7,Minimum Daily Revenue,"3,457.11"
8,Maximum Daily Revenue,"110,806.17"
9,Average Outlier Revenue,"84,805.36"



Outlier Dates and Revenue:


,Daily_Revenue
Transaction_Date,
2011-11-14,"110,806.17"
2011-09-20,"108,335.17"
2011-11-07,"82,628.60"
2010-12-07,"81,131.02"
2011-12-05,"79,665.11"
2011-12-08,"76,617.42"
2011-11-23,"75,496.41"
2011-10-05,"74,470.06"
2011-09-15,"74,098.28"


**Key Observations**
- **Nine trading days (2.95%) are identified as upper revenue outliers**, exceeding the IQR-based threshold of 73,724.87, while no lower outliers are detected.
- **Outlier days generate an average revenue of 84,805.36**, substantially exceeding the overall daily average of 32,109.50, indicating exceptionally strong sales activity on these dates.
- **November 14, 2011 records the highest daily revenue (110,806.17)**, followed by September 20, 2011 (108,335.17). Eight of the nine outlier dates occur between September and December 2011, suggesting a concentration of high-revenue activity toward the end of the observation period.

**Business Insights**
- **High-revenue outliers highlight periods of exceptional retail activity that may require additional inventory, staffing, and fulfillment capacity.**
- **The concentration of outliers in later months suggests that forecasting models should account for changing revenue levels and potential seasonal effects**, rather than treating all unusually high sales days as erroneous observations.

#### Analyzing Daily Transaction Volume

In [15]:
daily_transaction_volume = (
    merchandise_df.groupby("Transaction_Date")
    .size()
    .sort_index()
)

fig = go.Figure()

fig.add_trace(
    go.Histogram(
        x=daily_transaction_volume,
        nbinsx=30,
        name="Transaction Volume",
        marker=dict(
            color="#4C78A8",
            line=dict(color="white", width=1)
        ),
        opacity=0.85,
        hovertemplate=(
            "Daily Transaction Volume: %{x:,}"
            "<br>Number of Days: %{y}"
            "<extra></extra>"
        )
    )
)

fig.add_vline(
    x=daily_transaction_volume.mean(),
    line=dict(color="#E45756", width=2, dash="dash")
)

fig.add_vline(
    x=daily_transaction_volume.median(),
    line=dict(color="#2A9D8F", width=2, dash="dash")
)

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(color="#E45756", width=2, dash="dash"),
        name=f"Mean: {daily_transaction_volume.mean():,.0f}"
    )
)

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(color="#2A9D8F", width=2, dash="dash"),
        name=f"Median: {daily_transaction_volume.median():,.0f}"
    )
)

fig.update_layout(
    title=dict(
        text="Distribution of Daily Retail Transaction Volume",
        x=0.5,
        xanchor="center"
    ),
    xaxis_title="Daily Transaction Volume (Invoice Lines)",
    yaxis_title="Number of Days",
    template="plotly_white",
    width=1100,
    height=550,
    bargap=0.05,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    )
)

fig.show()

display(
    daily_transaction_volume.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    ).to_frame("Daily_Transaction_Volume")
)

,Daily_Transaction_Volume
count,305.00
mean,"1,767.21"
std,826.43
min,279.00
25%,"1,222.00"
50%,"1,566.00"
75%,"2,102.00"
90%,"2,937.20"
95%,"3,387.00"
99%,"4,173.52"


**Key Observations**
- **Daily transaction volume averages 1,767 invoice lines, with a median of 1,566**, indicating a positively skewed distribution influenced by days with unusually high transaction activity.
- **The middle 50% of trading days record between 1,222 and 2,102 invoice lines**, while a standard deviation of 826.43 highlights considerable variation in daily transaction activity.
- **Daily transaction volume ranges from 279 to 5,312 invoice lines**, with the 95th and 99th percentiles reaching 3,387 and 4,173.52, respectively, indicating occasional periods of exceptionally high activity.

**Business Insights**
- **Substantial variation in daily transaction volume highlights the importance of anticipating fluctuations in operational workload**, particularly for order processing, inventory handling, and fulfillment planning.
- **High-volume trading days may place additional pressure on retail operations**, making transaction volume a useful variable to investigate when analyzing daily revenue patterns and developing forecasting features.

#### Analyzing Daily Invoice Volume

In [16]:
daily_invoice_volume = (
    merchandise_df.groupby("Transaction_Date")["InvoiceNo"]
    .nunique()
    .sort_index()
)

mean_invoices = daily_invoice_volume.mean()
median_invoices = daily_invoice_volume.median()

fig = go.Figure()

fig.add_trace(
    go.Histogram(
        x=daily_invoice_volume,
        nbinsx=30,
        name="Daily Invoice Volume",
        marker=dict(
            color="#4C78A8",
            line=dict(color="white", width=1)
        ),
        opacity=0.85,
        hovertemplate=(
            "Daily Invoices: %{x:,}"
            "<br>Number of Days: %{y}"
            "<extra></extra>"
        )
    )
)

fig.add_vline(
    x=mean_invoices,
    line=dict(color="#E45756", width=2, dash="dash")
)

fig.add_vline(
    x=median_invoices,
    line=dict(color="#2A9D8F", width=2, dash="dash")
)

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(color="#E45756", width=2, dash="dash"),
        name=f"Mean: {mean_invoices:,.0f}"
    )
)

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(color="#2A9D8F", width=2, dash="dash"),
        name=f"Median: {median_invoices:,.0f}"
    )
)

fig.update_layout(
    title=dict(
        text="Distribution of Daily Retail Invoice Volume",
        x=0.5,
        xanchor="center"
    ),
    xaxis_title="Number of Unique Invoices",
    yaxis_title="Number of Days",
    template="plotly_white",
    width=1100,
    height=550,
    bargap=0.05,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    )
)

fig.show()

display(
    daily_invoice_volume.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    ).to_frame("Daily_Invoice_Volume")
)

,Daily_Invoice_Volume
count,305.00
mean,82.95
std,33.51
min,11.00
25%,61.00
50%,80.00
75%,102.00
90%,128.20
95%,147.00
99%,168.96


**Key Observations**
- **Daily invoice volume averages 82.95 unique invoices, with a median of 80**, indicating a slightly positively skewed distribution with occasional higher-activity trading days.
- **The middle 50% of trading days record between 61 and 102 unique invoices**, while a standard deviation of 33.51 reflects considerable variation in daily invoicing activity.
- **Daily invoice volume ranges from 11 to 214 unique invoices**, with the 95th and 99th percentiles reaching 147 and 168.96, respectively, highlighting occasional periods of exceptionally high invoice activity.

**Business Insights**
- **Variations in daily invoice volume highlight changing order-processing and billing workloads**, supporting the need for flexible operational capacity during busier trading periods.
- **Invoice volume provides a complementary measure of retail activity alongside invoice-line volume**, making it valuable for investigating whether daily revenue fluctuations are associated with changes in order frequency or invoice composition.

#### Analyzing Daily Units Sold

In [17]:
daily_units_sold = (
    merchandise_df.groupby("Transaction_Date")["Quantity"]
    .sum()
    .sort_index()
)

mean_units = daily_units_sold.mean()
median_units = daily_units_sold.median()

fig = go.Figure()

fig.add_trace(
    go.Histogram(
        x=daily_units_sold,
        nbinsx=30,
        name="Daily Net Units",
        marker=dict(
            color="#4C78A8",
            line=dict(color="white", width=1)
        ),
        opacity=0.85,
        hovertemplate=(
            "Daily Net Units: %{x:,.0f}"
            "<br>Number of Days: %{y}"
            "<extra></extra>"
        )
    )
)

fig.add_vline(
    x=mean_units,
    line=dict(color="#E45756", width=2, dash="dash")
)

fig.add_vline(
    x=median_units,
    line=dict(color="#2A9D8F", width=2, dash="dash")
)

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(color="#E45756", width=2, dash="dash"),
        name=f"Mean: {mean_units:,.0f}"
    )
)

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(color="#2A9D8F", width=2, dash="dash"),
        name=f"Median: {median_units:,.0f}"
    )
)

fig.update_layout(
    title=dict(
        text="Distribution of Daily Net Merchandise Units",
        x=0.5,
        xanchor="center"
    ),
    xaxis_title="Daily Net Merchandise Units",
    yaxis_title="Number of Days",
    template="plotly_white",
    width=1100,
    height=550,
    bargap=0.05,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    )
)

fig.show()

display(
    daily_units_sold.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    ).to_frame("Daily_Net_Units")
)

,Daily_Net_Units
count,305.00
mean,"16,938.51"
std,"8,948.51"
min,"-13,657.00"
25%,"10,708.00"
50%,"15,611.00"
75%,"21,188.00"
90%,"29,661.80"
95%,"33,453.40"
99%,"43,198.52"


**Key Observations**
- **Daily net merchandise quantities average 16,938.51 units, with a median of 15,611**, indicating a positively skewed distribution influenced by higher-volume trading days.
- **The middle 50% of trading days record between 10,708 and 21,188 net units**, while a standard deviation of 8,948.51 highlights substantial variation in daily merchandise movement.
- **Daily net quantities range from -13,657 to 46,140 units**, with the negative minimum indicating that returns or cancellations exceeded positive quantities on at least one trading day. The 95th and 99th percentiles reach 33,453.40 and 43,198.52 units, respectively.

**Business Insights**
- **Considerable fluctuations in merchandise quantities highlight the importance of anticipating changes in product movement**, supporting inventory replenishment, warehouse capacity, and fulfillment planning.
- **Negative net quantities demonstrate how returns and cancellations can distort daily sales activity**, making it important to distinguish net merchandise movement from gross sales when investigating revenue patterns and developing forecasting features.